In [1]:
import os
import string
import pickle
import numpy as np
import tensorflow as tf
from tqdm import tqdm

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer, Embedding, LSTM, Dense


2026-02-05 01:08:42.220453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770253722.497480      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770253722.575316      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770253723.201919      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770253723.201983      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770253723.201987      55 computation_placer.cc:177] computation placer alr

In [2]:
IMAGES_PATH = "/kaggle/input/flickr8k/Images"
CAPTIONS_FILE = "/kaggle/input/flickr8k/captions.txt"
FEATURES_PATH = "/kaggle/working/features.pkl"


In [3]:
def load_captions(filename):
    captions = {}
    with open(filename, 'r') as f:
        next(f)  # skip header
        for line in f:
            line = line.strip()
            if not line:
                continue
            img, caption = line.split(',', 1)
            caption = caption.lower().translate(
                str.maketrans('', '', string.punctuation)
            )
            caption = f"<start> {caption} <end>"
            captions.setdefault(img, []).append(caption)
    return captions

captions_dict = load_captions(CAPTIONS_FILE)

all_captions = []
for caps in captions_dict.values():
    all_captions.extend(caps)

print("Images:", len(captions_dict))
print("Captions:", len(all_captions))
print("Sample:", all_captions[0])


Images: 8091
Captions: 40455
Sample: <start> a child in a pink dress is climbing up a set of stairs in an entry way  <end>


In [4]:
tokenizer = Tokenizer(num_words=5000, oov_token="<unk>")
tokenizer.fit_on_texts(all_captions + ['<start>', '<end>'])

vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(c.split()) for c in all_captions)

print("Vocab size:", vocab_size)
print("Max caption length:", max_len)


Vocab size: 8830
Max caption length: 38


In [5]:
base_model = InceptionV3(weights="imagenet")
cnn_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer("mixed10").output
)

cnn_model.trainable = False


2026-01-27 03:31:10.423174: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
if os.path.exists(FEATURES_PATH):
    with open(FEATURES_PATH, "rb") as f:
        image_features = pickle.load(f)
else:
    image_features = {}
    image_files = [
        f for f in os.listdir(IMAGES_PATH)
        if f.lower().endswith(".jpg")
    ]

    BATCH_SIZE = 32

    for i in tqdm(range(0, len(image_files), BATCH_SIZE), desc="Extracting features"):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_imgs = []

        for fname in batch_files:
            img_path = os.path.join(IMAGES_PATH, fname)
            img = image.load_img(img_path, target_size=(299, 299))
            x = image.img_to_array(img)
            batch_imgs.append(x)

        batch_imgs = np.array(batch_imgs)
        batch_imgs = preprocess_input(batch_imgs)

        batch_features = cnn_model.predict(batch_imgs, verbose=0)
        batch_features = batch_features.reshape(batch_features.shape[0], 64, 2048)

        for fname, feat in zip(batch_files, batch_features):
            image_features[fname] = feat

    with open(FEATURES_PATH, "wb") as f:
        pickle.dump(image_features, f)

print("Features extracted:", len(image_features))


Extracting features: 100%|██████████| 253/253 [19:12<00:00,  4.55s/it]


Features extracted: 8091


In [7]:
class BahdanauAttention(Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, features, hidden):
        hidden_time = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_time))
        weights = tf.nn.softmax(self.V(score), axis=1)
        context = tf.reduce_sum(weights * features, axis=1)
        return context, weights


In [8]:
class RNNDecoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.units = units
        self.embedding = Embedding(vocab_size, embed_dim)
        self.lstm = LSTM(units, return_sequences=True, return_state=True)
        self.fc1 = Dense(units)
        self.fc2 = Dense(vocab_size)
        self.attention = BahdanauAttention(units)

    def call(self, x, features, hidden):
        context, _ = self.attention(features, hidden)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context, 1), x], axis=-1)
        output, h, c = self.lstm(x)
        x = self.fc1(output)
        x = tf.reshape(x, (-1, x.shape[2]))
        x = self.fc2(x)
        return x, h


In [9]:
embedding_dim = 256
units = 512

decoder = RNNDecoder(vocab_size, embedding_dim, units)
optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=5.0
    )
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def loss_function(y_true, y_pred):
    mask = tf.math.not_equal(y_true, 0)
    loss = loss_object(y_true, y_pred)
    mask = tf.cast(mask, loss.dtype)
    return tf.reduce_mean(loss * mask)


In [12]:
@tf.function
def train_step(img_tensor, target):
    loss = 0
    hidden = tf.zeros((1, units))
    dec_input = tf.expand_dims([tokenizer.word_index['start']], 1)


    with tf.GradientTape() as tape:
        for i in range(1, target.shape[1]):
            predictions, hidden = decoder(dec_input, img_tensor, hidden)
            loss += loss_function(target[:, i], predictions)
            dec_input = tf.expand_dims(target[:, i], 1)
    
    avg_loss = loss / tf.cast(target.shape[1], tf.float32)
    gradients = tape.gradient(loss, decoder.trainable_variables)
    optimizer.apply_gradients(zip(gradients, decoder.trainable_variables))
    return avg_loss


In [ ]:
from tqdm import tqdm
EPOCHS = 5
for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0

    for img_name in tqdm(captions_dict.keys(),
                         desc=f"Epoch {epoch+1}/{EPOCHS}",
                         leave=True):
        if img_name not in image_features:
            continue

        img_tensor = tf.expand_dims(image_features[img_name], 0)

        for cap in captions_dict[img_name]:
            seq = tokenizer.texts_to_sequences([cap])[0]
            seq = pad_sequences([seq], maxlen=max_len, padding="post")
            seq = tf.convert_to_tensor(seq)

            loss = train_step(img_tensor, seq)
            total_loss += loss
            steps += 1

    print(f"Epoch {epoch+1} Loss: {total_loss/steps:.4f}")


Epoch 1/2:  49%|████▉     | 3968/8091 [4:02:26<4:01:25,  3.51s/it]